In [1]:
import Pkg

In [2]:
Pkg.activate(".")
Pkg.add("AppleAccelerate")
Pkg.add("ThreadPinning")
Pkg.add("KrylovKit")
Pkg.add("BenchmarkTools")
Pkg.add("ProfileCanvas")
Pkg.instantiate()

  Activating project at `~/Documents/Research/2025-05_KrylovKitBenchmark`
   Resolving package versions...
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/Research/2025-05_

In [3]:
using Random
using KrylovKit: expintegrator, Arnoldi
using BenchmarkTools
using ProfileCanvas
using LinearAlgebra: norm

In [4]:
using ThreadPinning
pinthreads(:cores)
threadinfo()

Hostname: 	ophelia
CPU(s): 	1 x Apple M4 Max
CPU target: 	apple-m1
Cores: 		16 (16 CPU-threads)
Core kinds: 	4 "efficiency cores", 12 "performance cores".
NUMA domains: 	1 (16 cores each)

Unsupported OS: Won't be able to highlight Julia threads.

Julia threads: 	8

CPU socket 1
  0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15


# = Julia thread, # = >1 Julia thread, # = Efficiency core


In [5]:
filter(p -> contains(p[1], "THREAD"), ENV)

Dict{String, String} with 6 entries:
  "OPENBLAS_NUM_THREADS"   => "1"
  "VECLIB_MAXIMUM_THREADS" => "1"
  "OMP_NUM_THREADS"        => "1"
  "NUMEXPR_NUM_THREADS"    => "1"
  "MKL_NUM_THREADS"        => "1"
  "JULIA_NUM_THREADS"      => "8"

In [6]:
using AppleAccelerate

In [7]:
N = 100

100

In [8]:
"""Random complex matrix of dimension `N` with a spectral radius of approximately `ρ`."""
function random_matrix(N=N, ρ=1.0; rng=Random.GLOBAL_RNG)
    Δ = √(12 / N)
    X = Δ * (rand(rng, N, N) .- 0.5)
    Y = Δ * (rand(rng, N, N) .- 0.5)
    H = ρ * (X + Y * 1im) / √2
    return H
end

function random_hermitian_matrix(N=N, ρ=1.0; rng=Random.GLOBAL_RNG)
    Δ = √(12 / N)
    X = Δ * (rand(rng, N, N) .- 0.5)
    Y = Δ * (rand(rng, N, N) .- 0.5)
    Z = (X + Y * 1im) / √2
    H = ρ * (Z + Z') / (2 * √2)
    return H
end


"""Random normalized complex vector of dimension `N`"""
function random_state_vector(N=N; rng=Random.GLOBAL_RNG)
    Ψ = rand(rng, N) .* exp.((2π * im) .* rand(rng, N))
    Ψ ./= norm(Ψ)
    return Ψ
end

random_state_vector

In [9]:
struct Trajectory
    initial_state::Vector{ComplexF64}
    H::Matrix{ComplexF64}
    dt::Vector{Float64}
end

function Trajectory(;initial_state, H, nt)
    dt = rand(nt)
    N = length(initial_state)
    @assert size(H) == (N, N)
    Trajectory(initial_state, H, dt)
end

Trajectory

In [10]:
function propagate_traj(traj::Trajectory)
    # even if traj.H is Hermitian, we still use Arnoldi.
    # This propagation method is intended for non-Hermitian generators,
    # using a general matrix screws up the benchmark because the norm
    # of Ψ explodes.
    alg = Arnoldi()
    Ψ = traj.initial_state
    numops = 0
    for dt in traj.dt
        Ψ, info = expintegrator(traj.H, -1im * dt, (Ψ, ), alg)
        numops += info.numops
    end
    return numops
end

propagate_traj (generic function with 1 method)

In [11]:
traj100 = Trajectory(initial_state=random_state_vector(), H=random_hermitian_matrix(), nt=100);
propagate_traj(traj100)

3100

In [12]:
@benchmark propagate_traj(traj100)

BenchmarkTools.Trial: 188 samples with 1 evaluation.
 Range (min … max):  25.705 ms …  29.546 ms  ┊ GC (min … max): 1.12% … 10.83%
 Time  (median):     26.573 ms               ┊ GC (median):    1.97%
 Time  (mean ± σ):   26.595 ms ± 488.791 μs  ┊ GC (mean ± σ):  2.15% ±  1.39%

             ▂        █▃ ▂▃ ▃▃▃▇  ▇                             
  ▅▃▃▅▃▃▃▅▅█▇█▅▆██▆█████▅██▆████▅██▃▆██▇▅▆█▅▅▆▃▁▃▃▅▁▁▁▁▁▁▁▁▁▁▃ ▃
  25.7 ms         Histogram: frequency by time         27.8 ms <

 Memory estimate: 19.71 MiB, allocs estimate: 16100.

In [13]:
Base.GC.enable(false)
@benchmark propagate_traj(traj100)

BenchmarkTools.Trial: 179 samples with 1 evaluation.
 Range (min … max):  27.467 ms …  28.654 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     28.078 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   28.065 ms ± 200.621 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

                              ▃  ▃▁▄▁ █▁▁▂▄▁▃                   
  ▃▁▁▁▃▁▁▁▃▁▄▁▄▆▆▃▃▆▄▃▁▆▄▄▃▇▆▆█▇▆████████████▆▄▄▄▇▇▄▃▆▃▃▇▆▄▁▁▃ ▃
  27.5 ms         Histogram: frequency by time         28.5 ms <

 Memory estimate: 19.71 MiB, allocs estimate: 16100.

In [14]:
Base.GC.enable(true)

false

In [15]:
@profview propagate_traj(traj100)

ProfileCanvas.ProfileData(Dict{String, ProfileCanvas.ProfileFrame}("3" => ProfileCanvas.ProfileFrame("root", "", "", 0, 24, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("task_done_hook", "task.jl", "./task.jl", 694, 2, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("wait", "task.jl", "./task.jl", 1021, 2, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("poptask", "task.jl", "./task.jl", 1012, 2, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("trypoptask", "task.jl", "./task.jl", 1004, 1, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 143, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "abstractarray.jl", "./abstractarray.jl", 1315, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "essentials.jl", "./essentials.jl", 917, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])])])])])])]), "4" => ProfileCanvas.ProfileFrame("root", "", "", 0, 24, missing, 0x00, missing, ProfileCanvas.ProfileFrame[]), "1" => ProfileCanvas.ProfileFrame("root", "", "", 0, 24, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#15", "eventloop.jl", "/Users/goerz/.julia/packages/IJulia/XF6bn/src/eventloop.jl", 51, 15, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("eventloop", "eventloop.jl", "/Users/goerz/.julia/packages/IJulia/XF6bn/src/eventloop.jl", 14, 15, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("invokelatest", "essentials.jl", "./essentials.jl", 1052, 15, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#invokelatest#2", "essentials.jl", "./essentials.jl", 1055, 15, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("execute_request", "execute_request.jl", "/Users/goerz/.julia/packages/IJulia/XF6bn/src/execute_request.jl", 81, 15, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("softscope_include_string", "SoftGlobalScope.jl", "/Users/goerz/.julia/packages/SoftGlobalScope/u4UzH/src/SoftGlobalScope.jl", 65, 15, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("include_string", "loading.jl", "./loading.jl", 2734, 15, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("eval", "boot.jl", "./boot.jl", 430, 15, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("propagate_traj", "In[10]", "./In[10]", 10, 15, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("expintegrator", "expintegrator.jl", "/Users/goerz/.julia/packages/KrylovKit/diNbc/src/matrixfun/expintegrator.jl", 106, 15, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("expintegrator", "expintegrator.jl", "/Users/goerz/.julia/packages/KrylovKit/diNbc/src/matrixfun/expintegrator.jl", 189, 9, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("exp!", "dense.jl", "/Users/julia/.julia/scratchspaces/a66863c6-20e8-4ff4-8a62-49f30b1f605e/agent-cache/default-honeycrisp-HL2F7YQ3XH.0/build/default-honeycrisp-HL2F7YQ3XH-0/julialang/julia-release-1-dot-11/usr/share/julia/stdlib/v1.11/LinearAlgebra/src/dense.jl", 711, 3, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("*", "matmul.jl", "/Users/julia/.julia/scratchspaces/a66863c6-20e8-4ff4-8a62-49f30b1f605e/agent-cache/default-honeycrisp-HL2F7YQ3XH.0/build/default-honeycrisp-HL2F7YQ3XH-0/julialang/julia-release-1-dot-11/usr/share/julia/stdlib/v1.11/LinearAlgebra/src/matmul.jl", 130, 3, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("similar", "array.jl", "./array.jl", 372, 2, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("Array", "

In [16]:
traj1000 = Trajectory(initial_state=random_state_vector(), H=random_hermitian_matrix(), nt=1000);
propagate_traj(traj1000)

31000

In [17]:
@benchmark propagate_traj($traj1000)

BenchmarkTools.Trial: 19 samples with 1 evaluation.
 Range (min … max):  265.487 ms … 270.243 ms  ┊ GC (min … max): 2.00% … 2.41%
 Time  (median):     267.616 ms               ┊ GC (median):    2.12%
 Time  (mean ± σ):   267.644 ms ±   1.208 ms  ┊ GC (mean ± σ):  2.13% ± 0.20%

  ▁     ▁   ▁  ▁ ▁    ▁▁▁  ▁ ▁  ▁█   █   ▁  ▁           ▁     ▁  
  █▁▁▁▁▁█▁▁▁█▁▁█▁█▁▁▁▁███▁▁█▁█▁▁██▁▁▁█▁▁▁█▁▁█▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁█ ▁
  265 ms           Histogram: frequency by time          270 ms <

 Memory estimate: 197.11 MiB, allocs estimate: 161004.

In [18]:
@profview propagate_traj(traj1000)

ProfileCanvas.ProfileData(Dict{String, ProfileCanvas.ProfileFrame}("3" => ProfileCanvas.ProfileFrame("root", "", "", 0, 203, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("task_done_hook", "task.jl", "./task.jl", 694, 7, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("wait", "task.jl", "./task.jl", 1021, 7, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("poptask", "task.jl", "./task.jl", 1012, 7, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("trypoptask", "task.jl", "./task.jl", 1004, 3, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 138, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[]), ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 140, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("cong", "partr.jl", "./partr.jl", 23, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])]), ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 142, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "abstractarray.jl", "./abstractarray.jl", 1315, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "essentials.jl", "./essentials.jl", 916, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("length", "essentials.jl", "./essentials.jl", 11, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])])])]), ProfileCanvas.ProfileFrame("multiq_check_empty", "partr.jl", "./partr.jl", 179, 2, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])])]), ProfileCanvas.ProfileFrame("trypoptask", "task.jl", "./task.jl", 990, 1, missing, 0x10, missing, ProfileCanvas.ProfileFrame[])]), "4" => ProfileCanvas.ProfileFrame("root", "", "", 0, 203, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("task_done_hook", "task.jl", "./task.jl", 694, 8, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("wait", "task.jl", "./task.jl", 1021, 8, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("poptask", "task.jl", "./task.jl", 1012, 8, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("multiq_check_empty", "partr.jl", "./partr.jl", 179, 2, missing, 0x00, missing, ProfileCanvas.ProfileFrame[]), ProfileCanvas.ProfileFrame("multiq_check_empty", "partr.jl", "./partr.jl", 181, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[]), ProfileCanvas.ProfileFrame("trypoptask", "task.jl", "./task.jl", 1004, 2, missing, 0x10, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 138, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[]), ProfileCanvas.ProfileFrame("multiq_deletemin", "partr.jl", "./partr.jl", 142, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "abstractarray.jl", "./abstractarray.jl", 1315, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("getindex", "essentials.jl", "./essentials.jl", 916, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("length", "essentials.jl", "./essentials.jl", 11, 1, missing, 0x00, missing, ProfileCanvas.ProfileFrame[])])])])])])])])]), "1" => ProfileCanvas.ProfileFrame("root", "", "", 0, 203, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("#15", "eventloop.jl", "/Users/goerz/.julia/packages/IJulia/XF6bn/src/eventloop.jl", 51, 167, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("eventloop", "eventloop.jl", "/Users/goerz/.julia/packages/IJulia/XF6bn/src/eventloop.jl", 14, 167, missing, 0x00, missing, ProfileCanvas.ProfileFrame[ProfileCanvas.ProfileFrame("invokelatest", "essentials.jl", "./essentials.jl", 10